In [26]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import hstack, csr_matrix
import joblib

cols = ['id','label','statement','subject','speaker','job','state','party',
        'barely_true','false','half_true','mostly_true','pants_fire','context']

train = pd.read_csv('train.tsv', sep='\t', header=None, names=cols)
valid = pd.read_csv('valid.tsv', sep='\t', header=None, names=cols)
test  = pd.read_csv('test.tsv',  sep='\t', header=None, names=cols)

print(f"Train: {train.shape}, Valid: {valid.shape}, Test: {test.shape}")

Train: (10240, 14), Valid: (1284, 14), Test: (1267, 14)


In [27]:
fake_labels = ['false', 'pants-fire', 'barely-true']

def prepare(df):
    df = df.copy()
    
    # binary label
    df['binary_label'] = df['label'].apply(lambda x: 1 if x in fake_labels else 0)
    
    # speaker history — total lies on record
    df['false_count'] = (
        df['barely_true'].fillna(0) + 
        df['false'].fillna(0) + 
        df['pants_fire'].fillna(0)
    )
    df['true_count'] = (
        df['half_true'].fillna(0) + 
        df['mostly_true'].fillna(0)
    )
    
    # lie ratio — how often this speaker has lied before
    total = df['false_count'] + df['true_count']
    df['lie_ratio'] = df['false_count'] / total.replace(0, 1)
    
    # combined text — statement + speaker context
    df['combined_text'] = (
        df['statement'].fillna('') + ' ' +
        df['speaker'].fillna('') + ' ' +
        df['party'].fillna('') + ' ' +
        df['job'].fillna('')
    )
    
    return df

train = prepare(train)
valid = prepare(valid)
test  = prepare(test)

print("Label distribution:")
print(train['binary_label'].value_counts())

Label distribution:
binary_label
0    5752
1    4488
Name: count, dtype: int64


In [28]:
class TextFeatures(BaseEstimator, TransformerMixin):
    
    SENSATIONAL = re.compile(
        r'\b(BREAKING|EXCLUSIVE|SHOCKING|REVEALED|CONSPIRACY|'
        r'SECRET|BANNED|CENSORED|HOAX|URGENT|ALERT)\b',
        re.IGNORECASE
    )
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        features = []
        for text in X:
            text = str(text)
            words = text.split()
            features.append([
                len(text),                                      # total length
                len(words),                                     # word count
                text.count('!'),                                # exclamation marks
                text.count('?'),                                # question marks
                sum(1 for c in text if c.isupper()),            # caps count
                len(self.SENSATIONAL.findall(text)),            # sensational words
                sum(1 for w in words if w.isupper() and len(w) > 2),  # all-caps words
            ])
        return np.array(features, dtype=float)

In [29]:
# TF-IDF on combined text
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train['combined_text'])
X_valid_tfidf = tfidf.transform(valid['combined_text'])
X_test_tfidf  = tfidf.transform(test['combined_text'])

# custom text features
feat_extractor = TextFeatures()
X_train_feat = csr_matrix(feat_extractor.transform(train['combined_text']))
X_valid_feat = csr_matrix(feat_extractor.transform(valid['combined_text']))
X_test_feat  = csr_matrix(feat_extractor.transform(test['combined_text']))

# speaker history features
speaker_cols = ['false_count', 'true_count', 'lie_ratio']
X_train_spk = csr_matrix(train[speaker_cols].fillna(0).values)
X_valid_spk = csr_matrix(valid[speaker_cols].fillna(0).values)
X_test_spk  = csr_matrix(test[speaker_cols].fillna(0).values)

# combine everything
X_train = hstack([X_train_tfidf, X_train_feat, X_train_spk])
X_valid = hstack([X_valid_tfidf, X_valid_feat, X_valid_spk])
X_test  = hstack([X_test_tfidf,  X_test_feat,  X_test_spk])

# train
clf = LogisticRegression(max_iter=2000, C=1.0)
clf.fit(X_train, train['binary_label'])

# evaluate on validation set
preds = clf.predict(X_valid)
print("=== Validation Set ===")
print(classification_report(valid['binary_label'], preds, target_names=['Real', 'Fake']))

# evaluate on test set
preds_test = clf.predict(X_test)
print("=== Test Set ===")
print(classification_report(test['binary_label'], preds_test, target_names=['Real', 'Fake']))

=== Validation Set ===
              precision    recall  f1-score   support

        Real       0.72      0.80      0.76       668
        Fake       0.75      0.66      0.70       616

    accuracy                           0.73      1284
   macro avg       0.73      0.73      0.73      1284
weighted avg       0.73      0.73      0.73      1284

=== Test Set ===
              precision    recall  f1-score   support

        Real       0.77      0.82      0.79       714
        Fake       0.75      0.69      0.71       553

    accuracy                           0.76      1267
   macro avg       0.76      0.75      0.75      1267
weighted avg       0.76      0.76      0.76      1267



C:\Users\parul\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [30]:
joblib.dump(clf, 'fake_news_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(feat_extractor, 'text_features.pkl')

print("All three files saved!")
print("fake_news_model.pkl")
print("tfidf_vectorizer.pkl") 
print("text_features.pkl")

All three files saved!
fake_news_model.pkl
tfidf_vectorizer.pkl
text_features.pkl


In [33]:
def predict(text, speaker="unknown", party=""):
    combined = f"{text} {speaker} {party}"
    
    tfidf_vec = tfidf.transform([combined])
    feat_vec  = csr_matrix(feat_extractor.transform([combined]))
    spk_vec   = csr_matrix([[0, 0, 0.495]])  # zero history — changed from 0.5
    
    X = hstack([tfidf_vec, feat_vec, spk_vec])
    prob = clf.predict_proba(X)[0][1]
    label = "Fake" if prob > 0.5 else "Real"
    print(f"{label} ({prob:.1%} fake) — {text[:70]}")

predict("Pope Francis endorses Trump saying he is the only one who can save Christianity")
predict("NASA releases deepest infrared image of the universe from James Webb telescope")
predict("Scientists confirm drinking bleach cures all diseases instantly")

Real (31.5% fake) — Pope Francis endorses Trump saying he is the only one who can save Chr
Real (47.9% fake) — NASA releases deepest infrared image of the universe from James Webb t
Fake (56.4% fake) — Scientists confirm drinking bleach cures all diseases instantly


In [2]:
import joblib
import numpy as np
import re
from scipy.sparse import issparse, csr_matrix, hstack
from sklearn.base import BaseEstimator, TransformerMixin

# must define this before loading the pkl
class TextFeatures(BaseEstimator, TransformerMixin):
    SENSATIONAL = re.compile(
        r'\b(BREAKING|EXCLUSIVE|SHOCKING|REVEALED|CONSPIRACY|'
        r'SECRET|BANNED|CENSORED|HOAX|URGENT|ALERT)\b',
        re.IGNORECASE
    )
    def fit(self, X, y=None): return self
    def transform(self, X):
        features = []
        for text in X:
            text = str(text)
            words = text.split()
            features.append([
                len(text), len(words), text.count('!'), text.count('?'),
                sum(1 for c in text if c.isupper()),
                len(self.SENSATIONAL.findall(text)),
                sum(1 for w in words if w.isupper() and len(w) > 2),
            ])
        return np.array(features, dtype=float)

tfidf = joblib.load('tfidf_vectorizer.pkl')
clf   = joblib.load('fake_news_model.pkl')
feat  = joblib.load('text_features.pkl')

test_text = ["scientists confirm bleach cures diseases unknown"]
tfidf_vec = tfidf.transform(test_text)
feat_vec  = csr_matrix(feat.transform(test_text))
spk_vec   = csr_matrix(np.array([[0, 0, 0.495]]))
X = hstack([tfidf_vec, feat_vec, spk_vec])
prob = clf.predict_proba(X)[0][1]
print("Probability:", prob)

Probability: 0.5151489203772187


In [32]:
print("Average lie ratio in training set:", train['lie_ratio'].mean().round(3))

Average lie ratio in training set: 0.495
